<a href="https://colab.research.google.com/github/Kaique-Passos/FireDetection/blob/main/Comparacao_Modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparação entre os modelos

Lê os resultados dos treinos K-fold salvos no Google Drive (`MyDrive/histórico_treinamentos/<modelo>/<data_hora>/`), confere se todos os treinos usaram os mesmos folds e compara as métricas.

Não precisa de GPU: só lê arquivos CSV.

In [ ]:
import os

if os.path.isdir(os.path.join('/content/FireDetection', '.git')):
  %cd /content/FireDetection
  !git pull
  print('Repositório atualizado')
else:
  !git clone https://github.com/Kaique-Passos/FireDetection.git
  print('Repositório clonado')

os.chdir('/content/FireDetection')

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from utils.kfold import list_runs, load_kfold_results, check_same_folds, compare_kfold

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

history_root = '/content/drive/MyDrive/histórico_treinamentos'
print('Lendo treinos de:', history_root)

## Treinos disponíveis

Lista todos os treinos concluídos de cada modelo, com a configuração usada.

In [ ]:
treinos = []

for modelo in sorted(os.listdir(history_root)):
    for run in list_runs(history_root, modelo):
        with open(os.path.join(run, 'config.json')) as f:
            config = json.load(f)
        config['pasta'] = os.path.basename(run)
        treinos.append(config)

treinos = pd.DataFrame(treinos)
display(treinos)

## Escolha dos treinos

Por padrão usa o treino concluído mais recente de cada modelo. Para usar um treino específico, coloque o nome da pasta (por exemplo `'2026-10-18_21-40-12'`) em `pastas_escolhidas`.

In [ ]:
modelos = ['kutralnet_mobileoct', 'mobilenetv3']

# None = treino mais recente; ou o nome da pasta do treino
pastas_escolhidas = {
    'kutralnet_mobileoct': None,
    'mobilenetv3': None
}

runs = {}
for modelo in modelos:
    pasta = pastas_escolhidas.get(modelo)
    if pasta is None:
        disponiveis = list_runs(history_root, modelo)
        if not disponiveis:
            raise ValueError('Nenhum treino concluído de {}'.format(modelo))
        runs[modelo] = disponiveis[-1]
    else:
        runs[modelo] = os.path.join(history_root, modelo, pasta)

for modelo, run in runs.items():
    print('{}: {}'.format(modelo, run))

resultados = {modelo: load_kfold_results(run) for modelo, run in runs.items()}

## Verificação dos folds

Compara o `folds.csv` de cada treino (qual imagem ficou em qual fold de validação). Se todos forem iguais, os modelos foram treinados e validados exatamente nas mesmas partições. O hash SHA-256 serve como comprovação compacta: hashes iguais significam arquivos idênticos.

In [ ]:
mesmos_folds, relatorio_folds = check_same_folds(list(runs.values()))
display(relatorio_folds)

if not mesmos_folds:
    raise ValueError('Os treinos NÃO usaram os mesmos folds: a comparação pareada não é válida.')
print('Todos os treinos usaram os mesmos folds.')

## Comparação das métricas

Média ± desvio padrão de cada métrica nos 10 folds. Como os modelos usaram os mesmos folds, as amostras são pareadas: `t_test_p` é o teste t pareado e `wilcoxon_p` é o teste de Wilcoxon (não paramétrico). Um p < 0,05 indica diferença significativa.

In [ ]:
modelo_a, modelo_b = modelos[0], modelos[1]
res_a, res_b = resultados[modelo_a], resultados[modelo_b]

comparacao = compare_kfold(res_a, res_b, name_a=modelo_a, name_b=modelo_b)
display(comparacao)

comparacao_path = os.path.join(history_root, 'comparacoes',
                               '{}_vs_{}'.format(modelo_a, modelo_b),
                               '{}__{}'.format(os.path.basename(runs[modelo_a]),
                                               os.path.basename(runs[modelo_b])))
os.makedirs(comparacao_path, exist_ok=True)
comparacao.to_csv(os.path.join(comparacao_path, 'comparacao.csv'), index=False)
relatorio_folds.to_csv(os.path.join(comparacao_path, 'verificacao_folds.csv'), index=False)
print('Comparação salva em', comparacao_path)

In [ ]:
# boxplot da acurácia por fold (validação e teste)
metricas = [m for m in ['val_accuracy', 'test_accuracy'] if m in res_a.columns]
fig, axes = plt.subplots(1, len(metricas), figsize=(6 * len(metricas), 4), squeeze=False)

for ax, m in zip(axes[0], metricas):
    ax.boxplot([res_a[m], res_b[m]])
    ax.set_xticks([1, 2], [modelo_a, modelo_b])
    ax.set_title(m)
    ax.set_ylabel('Acurácia')

plt.tight_layout()
plt.savefig(os.path.join(comparacao_path, 'boxplot_acuracia.png'))
plt.show()

## Análise estatística

Espaço para a análise pedida pelo orientador.